# CIFAR-10 residual network under 5M parameters
Run top to bottom on Kaggle with a GPU. Attach the competition data and extract image archives first if necessary. Set the paths below for the labeled CSV and image directory. No pretrained weights or external training data are used.

The fixed, stratified 80/20 split is saved for reproducibility. Only training images receive random augmentation. This notebook runs only the small-batch overfit check using a disposable model. Full training can be added after the check passes.


In [ ]:
from pathlib import Path
import csv
import random
import numpy as np
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

SEED = 42
INPUT_ROOT = Path('/kaggle/input')
WORK_DIR = Path('/kaggle/working')
WORK_DIR.mkdir(parents=True, exist_ok=True)
# Set these explicitly if auto-discovery is ambiguous.
LABELS_CSV = None  # e.g. INPUT_ROOT / 'competition-name/trainLabels.csv'
TRAIN_IMAGE_DIR = None  # e.g. INPUT_ROOT / 'competition-name/train'

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## Load labels and freeze the split
Expected data: a CSV with `id,label` columns and one image per ID (for example `train/1.png`). Class names are mapped in sorted order and saved. The split uses a dedicated NumPy generator, independent of training randomness. Each class must have a sample count divisible by five so the requested 80/20 ratio is exact within every class.


In [ ]:
if LABELS_CSV is None:
    candidates = sorted(INPUT_ROOT.rglob('trainLabels.csv'))
    if len(candidates) != 1:
        raise ValueError(f'Set LABELS_CSV explicitly; found {candidates}')
    LABELS_CSV = candidates[0]
LABELS_CSV = Path(LABELS_CSV)
with LABELS_CSV.open(newline='') as f:
    reader = csv.DictReader(f)
    assert {'id', 'label'} <= set(reader.fieldnames or []), 'Expected id,label columns'
    rows = sorted(list(reader), key=lambda row: row['id'])
assert rows and len({r['id'] for r in rows}) == len(rows), 'IDs must be unique'
classes = sorted({r['label'] for r in rows})
assert len(classes) == 10, classes
class_to_idx = {name: i for i, name in enumerate(classes)}
labels = np.array([class_to_idx[r['label']] for r in rows], dtype=np.int64)
if TRAIN_IMAGE_DIR is None:
    candidates = sorted({p.parent for p in INPUT_ROOT.rglob(f"{rows[0]['id']}.png")})
    if len(candidates) != 1:
        raise ValueError(f'Set TRAIN_IMAGE_DIR explicitly; found {candidates}. Extract archives first.')
    TRAIN_IMAGE_DIR = candidates[0]
TRAIN_IMAGE_DIR = Path(TRAIN_IMAGE_DIR)
image_paths = [TRAIN_IMAGE_DIR / f"{r['id']}.png" for r in rows]
missing = [str(p) for p in image_paths if not p.is_file()]
assert not missing, f'Missing images, first examples: {missing[:5]}'

rng = np.random.default_rng(SEED)
train_indices, val_indices = [], []
for cls in range(10):
    indices = rng.permutation(np.flatnonzero(labels == cls))
    assert len(indices) % 5 == 0, 'Exact stratified 80/20 requires class sizes divisible by 5'
    n_val = len(indices) // 5
    val_indices.extend(indices[:n_val].tolist())
    train_indices.extend(indices[n_val:].tolist())
train_indices = np.array(sorted(train_indices))
val_indices = np.array(sorted(val_indices))
assert len(train_indices) * 5 == len(rows) * 4
assert len(val_indices) * 5 == len(rows)
assert not set(train_indices) & set(val_indices)
assert set(train_indices) | set(val_indices) == set(range(len(rows)))
with (WORK_DIR / 'split.csv').open('w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['id', 'label', 'split'])
    val_set = set(val_indices)
    for i, row in enumerate(rows):
        writer.writerow([row['id'], row['label'], 'validation' if i in val_set else 'training'])
print('Class mapping:', class_to_idx)
print('Training / validation:', len(train_indices), len(val_indices))
for cls, name in enumerate(classes):
    print(name, int((labels[train_indices] == cls).sum()), int((labels[val_indices] == cls).sum()))

In [ ]:
# Fixed CIFAR normalization constants; no statistics are fitted on validation data.
MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2470, 0.2435, 0.2616)
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class LabeledImages(Dataset):
    def __init__(self, indices, transform):
        self.indices = np.asarray(indices)
        self.transform = transform
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, index):
        i = int(self.indices[index])
        with Image.open(image_paths[i]) as image:
            image = image.convert('RGB')
            assert image.size == (32, 32), f'Unexpected image size: {image.size}'
            x = self.transform(image)
        return x, int(labels[i])

train_dataset = LabeledImages(train_indices, train_transform)
val_dataset = LabeledImages(val_indices, val_transform)
assert torch.equal(val_dataset[0][0], val_dataset[0][0]), 'Validation must be deterministic'


## Three-stage basic-block ResNet
The stem is a 3×3 stride-1 convolution with no max pooling. Stages have 64, 128, and 256 channels and three residual blocks each. Only the first blocks of stages 2 and 3 downsample, using stride 2 and a learned 1×1 shortcut. Global average pooling feeds a ten-class linear classifier.


In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.residual = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        self.shortcut = nn.Identity() if stride == 1 and in_channels == out_channels else nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.relu(self.residual(x) + self.shortcut(x))

class CIFARResNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.stage1 = nn.Sequential(BasicBlock(64, 64), BasicBlock(64, 64), BasicBlock(64, 64))
        self.stage2 = nn.Sequential(BasicBlock(64, 128, 2), BasicBlock(128, 128), BasicBlock(128, 128))
        self.stage3 = nn.Sequential(BasicBlock(128, 256, 2), BasicBlock(256, 256), BasicBlock(256, 256))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(256, 10)
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')
    def forward(self, x):
        x = self.stage3(self.stage2(self.stage1(self.stem(x))))
        return self.classifier(self.pool(x).flatten(1))

model = CIFARResNet()
parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
for name, module in model.named_children():
    print(f'{name}: {sum(p.numel() for p in module.parameters() if p.requires_grad):,}')
print(f'Trainable parameters: {parameter_count:,}')
assert parameter_count <= 5_000_000, f'Parameter budget exceeded: {parameter_count:,}'
with torch.no_grad():
    model.eval()
    assert model(torch.zeros(2, 3, 32, 32)).shape == (2, 10)

## Required small-batch overfit check (run on Kaggle)
Use four training examples per class with deterministic preprocessing. Repeatedly optimize the same 40 images with plain cross entropy, no augmentation, and no weight decay. Require 100% accuracy and cross entropy below 0.05 in evaluation mode within 500 steps. If the check fails, inspect labels, image loading, gradients, and learning rate before adding full training. The training step is reusable for later experiments.


In [ ]:
criterion = nn.CrossEntropyLoss()

def train_step(net, optimizer, images, targets):
    net.train()
    optimizer.zero_grad(set_to_none=True)
    logits = net(images)
    loss = criterion(logits, targets)
    assert torch.isfinite(loss), 'Non-finite training loss'
    loss.backward()
    optimizer.step()
    return float(loss.detach()), int((logits.detach().argmax(1) == targets).sum())

OVERFIT_PASSED = False

def check_small_batch():
    seed_everything(SEED)
    small_indices = np.concatenate([
        train_indices[labels[train_indices] == cls][:4] for cls in range(10)
    ])
    assert len(small_indices) == 40
    dataset = LabeledImages(small_indices, val_transform)
    images, targets = next(iter(DataLoader(dataset, batch_size=len(dataset), shuffle=False)))
    images, targets = images.to(device), targets.to(device)
    probe = CIFARResNet().to(device)
    optimizer = torch.optim.Adam(probe.parameters(), lr=1e-3, weight_decay=0)
    passed = False
    for step in range(1, 501):
        train_step(probe, optimizer, images, targets)
        if step == 1 or step % 10 == 0:
            probe.eval()
            with torch.no_grad():
                logits = probe(images)
                loss = criterion(logits, targets).item()
                accuracy = (logits.argmax(1) == targets).float().mean().item()
            print(f'Overfit step {step}: eval loss={loss:.4f}, accuracy={accuracy:.1%}')
            if accuracy == 1.0 and loss < 0.05:
                passed = True
                break
    assert passed, 'Small-batch overfit failed; resolve before full training.'
    return True

OVERFIT_PASSED = check_small_batch()